In [1]:
# ============================================================
# SETUP PATHS & IMPORTS
# ============================================================

import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Local path routing
# ------------------------------------------------------------
current_dir = os.getcwd()
workspace_root = current_dir

if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, "src")

sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# ------------------------------------------------------------
# Kaggle path routing
# ------------------------------------------------------------
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    for py_file in kaggle_input.rglob("*.py"):
        sys.path.insert(0, str(py_file.parent))

# ------------------------------------------------------------
# sklearn imports
# ------------------------------------------------------------
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    train_test_split,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# Optional Bayesian search imports
# ------------------------------------------------------------
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer
    SKOPT_AVAILABLE = True
except Exception:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    SKOPT_AVAILABLE = False

# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------
try:
    from src.ev_data_utils import load_ev_data
    from src.ev_baseline_utils import (
        EVFeatureExtractor,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded from src/")
except ImportError:
    from ev_data_utils import load_ev_data
    from ev_baseline_utils import (
        EVFeatureExtractor,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded flattened")

print(f"SKOPT_AVAILABLE: {SKOPT_AVAILABLE}")

✅ Imports loaded from src/
SKOPT_AVAILABLE: True


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

TARGET_COL = "Will_Buy_EV"

RANDOM_STATE = 42
N_SPLITS = 1                # Set to 1 for a single stratified fold, or >= 2 for K-Fold
HOLDOUT_SIZE = 0.4

MAKE_SUBMISSION = True

# ------------------------------------------------------------
# Search configuration
# ------------------------------------------------------------
SEARCH_MODE = "bayesian"        # "grid" or "bayesian"
N_ITER = 15                 # used only for Bayesian search
VERBOSE = 3
ERROR_SCORE = "raise"       # use "raise" for debugging

USE_BAYESIAN = SEARCH_MODE == "bayesian" and SKOPT_AVAILABLE

if SEARCH_MODE == "bayesian" and not SKOPT_AVAILABLE:
    print("⚠️ search_mode='bayesian' but scikit-optimize is unavailable.")
    print("⚠️ Falling back to grid search.")

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Search mode: {SEARCH_MODE}")
print(f"Using Bayesian search: {USE_BAYESIAN}")

Search mode: grid
Using Bayesian search: False


In [3]:
# ============================================================
# DATA LOADING
# ============================================================

train_df, test_df, data_source = load_ev_data(
    local_dir="data",
    train_name="train.csv",
    test_name="test.csv",
    sample_name="sample_ev.csv",
    target_col=TARGET_COL,
)

print(f"✅ Data source: {data_source}")
print(f"Train shape: {train_df.shape}")

if test_df is not None:
    print(f"Test shape: {test_df.shape}")

# ------------------------------------------------------------
# Target check
# ------------------------------------------------------------
if TARGET_COL not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train data.")

train_df = train_df.dropna(subset=[TARGET_COL]).copy()
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)

print("\nTarget distribution:")
print(train_df[TARGET_COL].value_counts())

assert set(train_df[TARGET_COL].unique()).issubset({0, 1}), (
    "Target must be binary 0/1. "
    "Coercion should happen inside ev_data_utils.load_ev_data."
)

# ------------------------------------------------------------
# Features / target
# ------------------------------------------------------------
X = train_df.drop(columns=[TARGET_COL], errors="ignore").copy()
y = train_df[TARGET_COL].astype(int).copy()

# ------------------------------------------------------------
# Optional holdout split
# ------------------------------------------------------------
if len(X) >= 50 and y.nunique() > 1:
    X_train, X_holdout, y_train, y_holdout = train_test_split(
        X,
        y,
        test_size=HOLDOUT_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    DO_HOLDOUT = True
else:
    X_train = X.copy()
    y_train = y.copy()
    X_holdout = None
    y_holdout = None
    DO_HOLDOUT = False

print(f"\nTrain rows used for fitting/search: {len(X_train)}")

if DO_HOLDOUT:
    print(f"Holdout rows: {len(X_holdout)}")
else:
    print("No holdout split.")

✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\evpurchase_kaggle\data
✅ Data source: Train file: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\evpurchase_kaggle\data\train.csv
Train shape: (668665, 15)
Test shape: (286571, 14)

Target distribution:
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

Train rows used for fitting/search: 468065
Holdout rows: 200600


In [4]:
# ============================================================
# RANDOM FOREST PIPELINE
# ============================================================

rf_pipeline = Pipeline(
    [
        (
            "extractor",
            EVFeatureExtractor(
                feature_set="multivariate",
                target_col=TARGET_COL,
                drop_id=True,
                impute_strategy="median",
                scale_numeric=False,  # Tree-based models don't require scaling
                onehot_categorical=True,
                add_derived_features=True,
            ),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

print("✅ Created Multivariate Random Forest pipeline.")

✅ Created Multivariate Random Forest pipeline.


In [5]:
# ============================================================
# COMPETITION SCORER
# ============================================================

scorer = make_competition_scorer(target_col=TARGET_COL)

print("✅ Using ROC AUC competition scorer.")

✅ Using ROC AUC competition scorer.


In [ ]:
# ============================================================
# GRID SEARCH SPACE
# ============================================================

RF_GRID_SPACE = {
    # Feature extraction parameters
    "extractor__add_derived_features": [True, False],
    "extractor__impute_strategy": ["median", "mean"],

    # Random Forest parameters
    "model__n_estimators": [5, 200],
    "model__max_depth": [None, 15, 25],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__class_weight": [None, "balanced"],
}

# ============================================================
# BAYESIAN SEARCH SPACE
# ============================================================

if SKOPT_AVAILABLE:
    RF_BAYESIAN_SPACE = {
        # Feature extraction parameters
        "extractor__add_derived_features": Categorical([True, False]),
        "extractor__impute_strategy": Categorical(["median", "mean"]),

        # Random Forest parameters
        "model__n_estimators": Integer(5, 200),
        "model__max_depth": Categorical([None, 10, 20, 50]),
        "model__min_samples_split": Integer(2, 10),
        "model__min_samples_leaf": Integer(1, 10),
        "model__class_weight": Categorical([None, "balanced"]),
    }
else:
    RF_BAYESIAN_SPACE = None

# ============================================================
# SELECT ACTIVE SEARCH SPACE
# ============================================================

if USE_BAYESIAN:
    RF_SEARCH_SPACE = RF_BAYESIAN_SPACE
    SEARCH_SPACE_KIND = "Bayesian"
else:
    RF_SEARCH_SPACE = RF_GRID_SPACE
    SEARCH_SPACE_KIND = "Grid"

print(f"✅ Active search-space type: {SEARCH_SPACE_KIND}")

✅ Active search-space type: Grid


In [7]:
# ============================================================
# CV OBJECT
# ============================================================

CAN_CV = (
    y_train.nunique() > 1
    and len(y_train) >= 4
    and int(y_train.value_counts().min()) >= 2
)

if CAN_CV:
    min_class_count = int(y_train.value_counts().min())
    
    if N_SPLITS == 1:
        # StratifiedKFold requires n_splits >= 2.
        # For a single stratified fold, we use StratifiedShuffleSplit.
        cv_object = StratifiedShuffleSplit(
            n_splits=1,
            test_size=HOLDOUT_SIZE,
            random_state=RANDOM_STATE,
        )
        print(f"✅ Using StratifiedShuffleSplit (1 stratified fold).")
    else:
        n_splits = max(2, min(N_SPLITS, min_class_count))
        cv_object = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        print(f"✅ Using StratifiedKFold with {n_splits} folds.")

else:
    cv_object = None
    print("⚠️ Not enough class diversity for stratified CV.")
    print("⚠️ Falling back to direct fit without search.")

✅ Using StratifiedShuffleSplit (1 stratified fold).


In [8]:
# ============================================================
# SEARCH EXECUTION
# ============================================================

if CAN_CV:

    if USE_BAYESIAN:
        rf_search = BayesSearchCV(
            estimator=rf_pipeline,
            search_spaces=RF_SEARCH_SPACE,
            n_iter=N_ITER,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )
    else:
        rf_search = GridSearchCV(
            estimator=rf_pipeline,
            param_grid=RF_SEARCH_SPACE,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

    print(f"🚀 Running {SEARCH_SPACE_KIND} search for Random Forest model...")
    rf_search.fit(X_train, y_train)

    BEST_PIPELINE = rf_search.best_estimator_
    SEARCH_SCORE = float(rf_search.best_score_)

    print("\nBest params:")
    for k, v in rf_search.best_params_.items():
        print(f"  {k}: {v}")

else:
    print("⚠️ Skipping search and fitting pipeline directly.")
    rf_pipeline.fit(X_train, y_train)
    rf_search = None
    BEST_PIPELINE = rf_pipeline
    SEARCH_SCORE = competition_score(
        y_train,
        BEST_PIPELINE.predict_proba(X_train)[:, 1],
    )

search_results = pd.DataFrame(
    [
        {
            "model": "random_forest",
            "search_type": SEARCH_SPACE_KIND,
            "search_score_roc_auc": SEARCH_SCORE,
        }
    ]
)

print("\nSearch Results:")
print(search_results)

🚀 Running Grid search for Random Forest model...
Fitting 1 folds for each of 192 candidates, totalling 192 fits
[CV 1/1] END extractor__add_derived_features=True, extractor__impute_strategy=median, model__class_weight=None, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100;, score=(train=1.000, test=0.934) total time=  34.7s
[CV 1/1] END extractor__add_derived_features=True, extractor__impute_strategy=median, model__class_weight=None, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=200;, score=(train=1.000, test=0.935) total time= 1.3min
[CV 1/1] END extractor__add_derived_features=True, extractor__impute_strategy=median, model__class_weight=None, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100;, score=(train=1.000, test=0.935) total time=  41.8s


KeyboardInterrupt: 

In [ ]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================

if DO_HOLDOUT:
    rf_holdout_score = competition_score(
        y_holdout,
        BEST_PIPELINE.predict_proba(X_holdout)[:, 1],
    )

    print("\nHoldout ROC AUC:")
    print(f"Random Forest: {rf_holdout_score:.4f}")

    FINAL_SCORE = rf_holdout_score
else:
    FINAL_SCORE = SEARCH_SCORE

print(f"\n🏆 Final ROC AUC Score: {FINAL_SCORE:.4f}")

In [ ]:
# ============================================================
# SAVE CV RESULTS WITH PARAMETERS
# ============================================================

if 'rf_search' in locals() and rf_search is not None:
    rf_cv_df = pd.DataFrame(rf_search.cv_results_)
    rf_cv_df['model_architecture'] = 'random_forest'
    
    # Move the architecture and score columns to the front for readability
    cols_to_front = ['model_architecture', 'mean_test_score', 'std_test_score', 'params']
    other_cols = [c for c in rf_cv_df.columns if c not in cols_to_front]
    rf_cv_df = rf_cv_df[cols_to_front + other_cols]
    
    cv_path = results_dir / f"rf_cv_results_with_params_{timestamp}.csv"
    rf_cv_df.to_csv(cv_path, index=False)
    print(f"✅ CV results with parameters saved to: {cv_path}")
else:
    print("⚠️ No search object found. Skipped saving CV results.")

In [ ]:
# ============================================================
# HOLDOUT ERROR ANALYSIS BY PROMINENT CATEGORIES
# ============================================================

if DO_HOLDOUT and X_holdout is not None:
    print("🚀 Generating holdout error analysis...")
    
    # 1. Get predictions from the best pipeline
    holdout_proba = BEST_PIPELINE.predict_proba(X_holdout)[:, 1]
    holdout_preds = (holdout_proba >= 0.5).astype(int)
    
    # 2. Create a master evaluation dataframe
    eval_df = X_holdout.copy()
    eval_df['true_label'] = y_holdout.values
    eval_df['pred_label'] = holdout_preds
    eval_df['pred_proba'] = holdout_proba
    eval_df['is_correct'] = eval_df['true_label'] == eval_df['pred_label']
    
    # 3. Define the prominent categorical/ordinal features to analyze
    prominent_categories = [
        'Gender', 
        'City_Type', 
        'Current_Car_Type', 
        'Home_Charging_Possible', 
        'Subsidy_Available', 
        'Range_Anxiety_Level'
    ]
    
    # Filter to only categories that actually exist in the dataset
    valid_categories = [c for c in prominent_categories if c in eval_df.columns]
    
    breakdown_list = []
    
    # 4. Group by each category and calculate correct/wrong metrics
    for cat in valid_categories:
        grp = eval_df.groupby(cat).agg(
            total_samples=('is_correct', 'count'),
            correct_predictions=('is_correct', 'sum'),
            wrong_predictions=('is_correct', lambda x: (x == False).sum()),
            accuracy=('is_correct', 'mean'),
            actual_yes_count=('true_label', 'sum'),
            predicted_yes_count=('pred_label', 'sum')
        ).reset_index()
        
        grp['feature_name'] = cat
        grp.rename(columns={cat: 'feature_value'}, inplace=True)
        breakdown_list.append(grp)
        
    if breakdown_list:
        breakdown_df = pd.concat(breakdown_list, ignore_index=True)
        
        # Reorder columns for easy reading
        cols = [
            'feature_name', 'feature_value', 'total_samples', 
            'correct_predictions', 'wrong_predictions', 'accuracy', 
            'actual_yes_count', 'predicted_yes_count'
        ]
        breakdown_df = breakdown_df[cols]
        
        # Save the breakdown to CSV
        breakdown_path = results_dir / f"rf_holdout_category_breakdown_{timestamp}.csv"
        breakdown_df.to_csv(breakdown_path, index=False)
        print(f"✅ Holdout category breakdown saved to: {breakdown_path}")
        
        # Display in notebook
        display(breakdown_df)
        
    else:
        print("⚠️ No valid categorical features found in X_holdout for breakdown.")
        
    # 5. Save the full row-level predictions for deep-dive debugging
    row_level_path = results_dir / f"rf_holdout_row_level_predictions_{timestamp}.csv"
    eval_df.to_csv(row_level_path, index=False)
    print(f"\n✅ Full row-level holdout predictions (with true/pred labels) saved to: {row_level_path}")

else:
    print("⚠️ Holdout evaluation was not performed or X_holdout is unavailable.")